In [63]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from bs4 import BeautifulSoup
from tqdm import tqdm
import re

from bs4 import XMLParsedAsHTMLWarning
import warnings

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

## **1. SCRAPER SETUP**

In [64]:
def bs4_extractor(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    return re.sub(r"\n\n+", "\n\n", soup.text).strip()

In [ ]:
website_url = "https://pathway.com/developers/user-guide/introduction/pathway-overview/"
loader = RecursiveUrlLoader(
    website_url, extractor=bs4_extractor)

## **2. STREAMING & CHUNKING**

In [66]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200)

In [67]:
all_chunks = []

# We use a loop to handle documents one-by-one (Streaming)
for doc in tqdm(loader.lazy_load(), bar_format="[INFO] {n_fmt} Pages Scraped."):

    # split_documents returns a LIST of chunks for this specific page
    chunks = text_splitter.split_documents([doc])

    # We use EXTEND to flatten those chunks into our main list
    all_chunks.extend(chunks)

print(f"[INFO] Created {len(all_chunks)} chunks.")

[INFO] 45 Pages Scraped.

[INFO] Created 1314 chunks.


In [68]:
all_chunks[0]

Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/overview', 'content_type': 'text/html; charset=utf-8', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain is an open source framework with a prebuilt agent architecture and integrations for any model or tool—so you can build agents that adapt as fast as the ecosystem evolves', 'language': 'en'}, page_content='LangChain overview - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-age

## **3. EMBEDDINGS & VECTOR STORE**

In [69]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")

# Pass the list of chunks directly - metadata is preserved automatically!
vector_store = InMemoryVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings
)

## **4. RETRIEVER**

In [70]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 10,
        "lambda_mult": 0.5
    }
)

In [71]:
# docs = retriever.invoke("What is LangSmith?")

# context = "\n\n".join(
#     [doc.page_content for doc in docs]
# )

# print(context)

In [72]:
llm = ChatOllama(model="deepseek-v3.1:671b-cloud", temperature=0.2)

## **5. PROMPT TEMPLATE**

In [73]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system",

         """
         You are a helpful AI assistant.
        Use ONLY the provided context to answer the question.
        DONT MAKE UP ANSWERS.

        If the answer is not present in the context,
        say: "I could not find the answer in the document."
        """),

        ("human",
         """
        Context:
        {context}

        Question:
        {question}
        """)
    ]
)

## **Final Implementation**

In [74]:
while True:
    query = input("You : ")
    if query == "0":
        break

    docs = retriever.invoke(query)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    final_prompt = prompt.invoke({
        "context": context,
        "question": query
    })

    response = llm.invoke(final_prompt)

    print(f"\n AI: {response.content}")


 AI: I could not find the answer in the document.

 AI: Based solely on the provided context, LangSmith is a platform released by LangChain Inc. that provides observability and evals to help build reliable AI agents. It helps trace requests, evaluate outputs, test prompts, and manage deployments in one place. LangSmith works with any agent stack and offers hosting options including self-hosted and hybrid.

 AI: I could not find the answer in the document.

 AI: I could not find the answer in the document.

 AI: Based on the provided context, LangGraph is a low-level orchestration framework and runtime for building, managing, and deploying long-running, stateful agents. It is built by LangChain Inc but can be used without LangChain. Its public interface draws inspiration from NetworkX, and it is inspired by Pregel and Apache Beam.

 AI: Based on the provided context, LangGraph is a low-level orchestration framework and runtime for building, managing, and deploying agents. LangChain age